In [1]:
import pandas as pd
from itertools import islice
from pyexpat import features
import torch
import numpy as np
from torchvision import transforms
import pandas as pd
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import skew
from sklearn.preprocessing import MinMaxScaler


In [3]:
train_df = pd.read_csv('./dataset/train.csv')
det_df = pd.read_csv('./dataset/train_det.csv')
gt_df = pd.read_csv('./dataset/train_gt.csv')

In [5]:
test_df = pd.read_csv('./dataset/test.csv')
test_det_df = pd.read_csv('./dataset/test_det.csv')

In [6]:
train_df.head()

,Unnamed: 0,video_id,video_name,image_width,image_height,frame_id,image_path
0,0,0,ADL-Rundle-6,1920,1080,0,./dataset/train/ADL-Rundle-6/img1/000001.jpg
1,1,0,ADL-Rundle-6,1920,1080,1,./dataset/train/ADL-Rundle-6/img1/000002.jpg
2,2,0,ADL-Rundle-6,1920,1080,2,./dataset/train/ADL-Rundle-6/img1/000003.jpg
3,3,0,ADL-Rundle-6,1920,1080,3,./dataset/train/ADL-Rundle-6/img1/000004.jpg
4,4,0,ADL-Rundle-6,1920,1080,4,./dataset/train/ADL-Rundle-6/img1/000005.jpg


In [7]:
det_df.head()

,Unnamed: 0,frame_id,object_id,x_coordinate,y_coordinate,width,height,confidence,x_init,y_init,z_init,video_id
0,0,0,-1,1689,385,146.620,332.710,67.567,-1.0,-1.0,-1.0,0
1,1,0,-1,1303,503,61.514,139.590,29.439,-1.0,-1.0,-1.0,0
2,2,0,-1,1258,569,40.123,91.049,19.601,-1.0,-1.0,-1.0,0
3,3,0,-1,31,525,113.370,257.270,17.013,-1.0,-1.0,-1.0,0
4,4,0,-1,1800,483,94.660,214.810,11.949,-1.0,-1.0,-1.0,0


In [ ]:
import torch
import torchvision
from torchvision.transforms import functional as F
from PIL import Image

# 1. Load the pre-trained Faster R-CNN model
# Weights="DEFAULT" loads the best available weights (typically COCO v1)
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
model.eval()  # Set to evaluation mode for inference

# 2. Load and preprocess an image
image = Image.open("sample_image.jpg").convert("RGB")
image_tensor = F.to_tensor(image).unsqueeze(0)  # Convert to tensor and add batch dim

# 3. Perform inference
with torch.no_grad():
    predictions = model(image_tensor)

# 4. Extract results
# Output is a list of dicts (one per image) containing boxes, labels, and scores
results = predictions[0]
boxes = results['boxes']   # Bounding boxes [x1, y1, x2, y2]
labels = results['labels'] # Class indices
scores = results['scores'] # Confidence scores

# Example: Filter for high-confidence detections
threshold = 0.8
high_conf_indices = [i for i, score in enumerate(scores) if score > threshold]
final_boxes = boxes[high_conf_indices]

In [ ]:
CONFIG = {
    'resize_dim': (640, 640),
}

In [8]:
class Preprocess:

    def _process_image(self, row: dict):
        video_id = row['video_id']
        image_path = row['image_path']

        img = cv.imread(image_path)
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

        ih, iw = row['image_height'], row['image_width']
        tw, th = CONFIG["resize_dim"]
        scale = min(tw / iw, th / ih)
        new_width, new_height = int(iw * scale), int(ih * scale)
        resized_image = cv.resize(image, (new_width, new_height), interpolation=cv.INTER_AREA)

        preprocess = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
        ])

        input_tensor = preprocess(resized_image)
        return features


    def process_dataset(self, df: pd.DataFrame):
        rows = df.reset_index().to_dict('records')
        for row in rows:
            self._process_image(row)
